<a href="https://colab.research.google.com/github/Odjewdheij/text-analysis-final/blob/text-analysis-final/final_project_shuyuwu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Measuring News Sentiment and Policy Topic Exposure for Major Firms**

## 1. Research Question and Policy Relevance

Financial news plays a central role in shaping how markets and the public interpret developments in the technology, financial, and energy sectors—especially during a period defined by rapid AI expansion, shifting macroeconomic conditions, and volatile energy markets. This project analyzes seven large, systemically important firms: **Apple (AAPL), Microsoft (MSFT), NVIDIA (NVDA), Tesla (TSLA), JPMorgan Chase (JPM), Goldman Sachs (GS), and Exxon Mobil (XOM).**

**Research question:**
*How does news sentiment vary across these firms, and how strongly are their news narratives associated with three policy-relevant themes: macroeconomic policy, AI/technology policy, and energy/commodity policy?*

This question matters for policymakers for several reasons:

* **Macroeconomic policy:**
  Sentiment toward financial firms influences expectations about monetary policy, financial stability, and credit conditions.

* **AI and technology governance:**
  Negative or skeptical sentiment around tech companies—especially NVIDIA, Microsoft, and Tesla—may indicate public concerns about an *AI bubble*, data governance risk, or uncertainty about future regulation.

* **Energy and climate policy:**
  News coverage of energy firms like ExxonMobil reflects ongoing debates around energy security, carbon policy, and commodity-cycle volatility.

To answer this question, the project combines:

1. **API-based financial news collection**,
2. **web scraping of full article text**,
3. **VADER sentiment scoring**, and
4. **a dictionary-based topic-exposure model** measuring coverage related to *macro*, *AI*, and *energy* themes.

In [1]:
!pip -q install requests beautifulsoup4 lxml yfinance nltk plotly kaleido

import re
from collections import Counter
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import requests
from bs4 import BeautifulSoup
import yfinance as yf

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import plotly.express as px
import plotly.io as pio

# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('vader_lexicon')
nltk.download('punkt_tab')

# Display options for pandas
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)

# List of tickers analyzed in this project
TICKERS = ["AAPL", "MSFT", "NVDA", "JPM", "GS", "XOM", "TSLA"]

print("✅ Imports and NLTK resources ready.")
print(f"✅ We will collect financial news for {len(TICKERS)} tickers:", TICKERS)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


✅ Imports and NLTK resources ready.
✅ We will collect financial news for 7 tickers: ['AAPL', 'MSFT', 'NVDA', 'JPM', 'GS', 'XOM', 'TSLA']


[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 2. Data collection: building a clean financial news corpus

My goal in this step is to build a high-quality news dataset for seven major firms  
(Apple, Microsoft, NVIDIA, Tesla, JPMorgan, Goldman Sachs, and Exxon Mobil).

Instead of taking every raw URL from the API, I use a three-stage pipeline:

1. **Query design** – search by both company name and ticker (e.g., `"Apple" OR AAPL`).  
2. **Metadata-level filtering** – remove paywalled/aggregator domains, non-news pages, and ad-like content before scraping.  
3. **Body-text relevance filtering** – after scraping, keep only articles whose full text is long enough and clearly related to the target firm based on a finance-specific dictionary.


### 2.1 Query design: company name + ticker

Many financial articles mention companies by name, ticker, or both.  
To capture these systematically, I define a set of query keywords for each firm  
—for example `"Apple" OR AAPL` or `"Goldman Sachs" OR GS`—and pass this combined query to the NewsAPI.


In [2]:
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Mapping from ticker to the keywords used in the NewsAPI query
TICKER_KEYWORDS = {
    "AAPL": ["Apple", "AAPL"],
    "MSFT": ["Microsoft", "MSFT"],
    "NVDA": ["NVIDIA", "NVDA"],
    "TSLA": ["Tesla", "TSLA"],
    "JPM":  ["JPMorgan", "JPM"],
    "GS":   ["Goldman Sachs", "GS"],
    "XOM":  ["Exxon Mobil", "XOM"],
}

NEWSAPI_KEY = "022fb134ae0549258ae4a52d89a994f4"   # <-- put your own key
NEWS_URL = "https://newsapi.org/v2/everything"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

def build_query(keywords):
    """
    Build a NewsAPI query string, e.g. ["Apple", "AAPL"] -> '"Apple" OR AAPL'.
    This lets me search for both the company name and the ticker.
    """
    parts = []
    for kw in keywords:
        if " " in kw:
            parts.append(f'"{kw}"')   # quote multi-word names
        else:
            parts.append(kw)
    return " OR ".join(parts)

### 2.2 Metadata-level filtering: removing low-value and ad-like content

Raw API results contain many URLs that are not useful for text analysis:

- Paywalled or low-value aggregators (e.g., **The Fly**, **Biztoc**, **Slashdot**).
- Non-news pages such as driver downloads, manuals, app pages, and shopping deals.
- Articles with missing URLs.

Before scraping any HTML, I:

1. Drop rows with missing URLs.  
2. Exclude known low-value domains: `thefly.com`, `biztoc.com`, `slashdot.org`.  
3. Remove titles and URLs that contain obvious non-news or ad-like keywords such as  
   `driver`, `download`, `manual`, `support`, `deal/deals`, `sale`, `discount`, `promo`, or `sponsored`.  
4. Deduplicate by a normalized title string so repeated headlines are only counted once.

I also print article counts at each step to see where the sample shrinks.


In [3]:
# Non-news / ad-like keywords in titles and URLs
BAD_TITLE_KEYWORDS = [
    "driver", "drivers", "download", "downloads",
    "manual", "user guide", "installation", "install",
    "setup", "how to", "faq", "support page", "release notes",
    "patch notes", "coupon", "deal", "deals",
    # ad-like words
    "sponsored", "promotion", "promo", "advertisement",
    "limited time", "sale", "discount"
]

BAD_URL_KEYWORDS = [
    "support", "download", "drivers", "manual",
    "help", "faq", "store", "shop", "community", "forum",
    "deals", "coupon"
]

BAD_TITLE_PAT = re.compile("|".join(re.escape(k) for k in BAD_TITLE_KEYWORDS))
BAD_URL_PAT   = re.compile("|".join(re.escape(k) for k in BAD_URL_KEYWORDS))

# Domains to exclude completely (paywalls / aggregators)
BAD_DOMAINS = ["thefly.com", "biztoc.com", "slashdot.org"]


def fetch_candidates_for_ticker(ticker, keywords, max_pages=10, page_size=100):
    """
    Fetch up to max_pages * page_size candidate articles for a given ticker.

    Steps:
      1. Query NewsAPI using both company name and ticker.
      2. Drop rows with missing URLs.
      3. Remove paywalled / low-value domains (thefly.com, biztoc.com, slashdot.org).
      4. Remove obvious non-news or ad-like pages by title and URL keywords.
      5. Deduplicate by normalized title.

    I print the number of articles after each filter so it is clear
    why some tickers end up with fewer usable items.
    """
    all_records = []
    query = build_query(keywords)

    print(f"\n🔎 Fetching metadata for {ticker} with query: {query}")

    for page in range(1, max_pages + 1):
        params = {
            "q": query,
            "language": "en",
            "pageSize": page_size,
            "page": page,
            "sortBy": "publishedAt",
            "apiKey": NEWSAPI_KEY,
            "searchIn": "title,description,content",
        }

        resp = requests.get(NEWS_URL, params=params)
        print(f"  Page {page} status:", resp.status_code)
        data = resp.json()
        articles = data.get("articles", [])
        if not articles:
            break

        for article in articles:
            title = (article.get("title") or "").strip()
            desc = article.get("description") or ""
            url = article.get("url") or ""
            published_at = article.get("publishedAt") or ""
            source_name = ""
            if isinstance(article.get("source"), dict):
                source_name = article["source"].get("name", "")

            all_records.append({
                "ticker": ticker,
                "source_name": source_name,
                "title": title,
                "description": desc,
                "url": url,
                "publishedAt": published_at,
            })

        # If fewer than page_size results, we are likely at the end of the result set
        if len(articles) < page_size:
            break
        time.sleep(0.7)  # be polite to the API

    if not all_records:
        print(f"⚠️ No articles found for {ticker}")
        return pd.DataFrame()

    df = pd.DataFrame(all_records)
    print(f"  Raw API articles: {len(df)}")

    # Drop missing URLs
    df = df.dropna(subset=["url"])
    print(f"  After dropping missing URLs: {len(df)}")

    # Normalize for filtering
    df["title_norm"]   = df["title"].str.lower().str.strip()
    df["title_lower"]  = df["title"].str.lower()
    df["url_lower"]    = df["url"].str.lower()
    df["source_lower"] = df["source_name"].str.lower()

    # Filter 0: remove BAD_DOMAINS (The Fly, Biztoc, Slashdot)
    mask_bad_domain = pd.Series(False, index=df.index)
    for dom in BAD_DOMAINS:
        root = dom.split(".")[0]
        mask_bad_domain |= df["url_lower"].str.contains(dom, na=False)
        mask_bad_domain |= df["source_lower"].str.contains(root, na=False)
    df = df[~mask_bad_domain].copy()
    print(f"  After removing bad domains: {len(df)}")

    # Filter 1: remove obvious non-news / ad-like pages by title and URL
    mask_bad_title = df["title_lower"].str.contains(BAD_TITLE_PAT, na=False)
    mask_bad_url   = df["url_lower"].str.contains(BAD_URL_PAT, na=False)
    df = df[~(mask_bad_title | mask_bad_url)].copy()
    print(f"  After removing non-news/ad-like pages: {len(df)}")

    # Filter 2: deduplicate by normalized title
    df = df.drop_duplicates(subset=["title_norm"], keep="first")
    print(f"  After title deduplication: {len(df)}")

    return df

### 2.3 Scraping and body-text relevance filtering

Title keywords are too strict: many good articles mention the firm mainly in the body,  
especially for analyst commentary and market summaries.

So instead of filtering by title, I scrape each candidate URL and apply a **body-text relevance test**:

1. Extract all `<p>` tags and join them into a single string.  
2. Require at least **250 characters** and at least **20 unique words** (to avoid empty or trivial pages).  
3. Use a firm-specific **industry dictionary** (e.g., “iPhone”, “Mac”, “AAPL” for Apple; “GPU”, “chip”, “AI” for NVIDIA) and require at least **two matches** in the article text.

For each ticker, I iterate through candidates and stop once I have **up to 30 valid, relevant articles**.


In [4]:
# Firm- and industry-specific dictionaries for body-text relevance
INDUSTRY_DICT = {
    "AAPL": ["apple", "iphone", "ipad", "tim cook", "aapl"],
    "MSFT": ["microsoft", "azure", "office", "msft"],
    "NVDA": ["nvidia", "gpu", "gpus", "chip", "chips", "ai", "semiconductor", "nvda"],
    "TSLA": ["tesla", "ev", "electric vehicle", "autopilot", "elon musk", "tsla"],
    "JPM":  ["jpmorgan", "jpm", "bank", "loan", "trading", "finance"],
    "GS":   ["goldman", "goldman sachs", "gs", "investment bank", "trading", "wall street"],
    "XOM":  ["exxon", "exxon mobil", "xom", "oil", "gas", "energy", "crude"],
}


def scrape_article_text(url):
    """
    Scrape a single article and return the concatenated text from all <p> tags.
    Returns an empty string on failure.
    """
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            print(f"  ⚠️ Failed {r.status_code}: {url}")
            return ""
        soup = BeautifulSoup(r.text, "html.parser")
        paras = [p.get_text(strip=True) for p in soup.find_all("p")]
        return " ".join([p for p in paras if p])
    except Exception as e:
        print(f"  ⚠️ Scrape error: {e}")
        return ""


def body_relevant(text, ticker):
    """
    Determine if an article is relevant based on its body text.
    Conditions:
      - length >= 250 characters
      - at least 20 unique word tokens
      - at least 2 matches from the company/industry dictionary
    """
    if not text or len(text) < 250:
        return False

    tokens = re.findall(r"[a-zA-Z]+", text.lower())
    if len(set(tokens)) < 20:
        return False

    dict_words = INDUSTRY_DICT.get(ticker, [])
    lower_text = text.lower()
    matches = sum(word in lower_text for word in dict_words)

    return matches >= 2

For each ticker, I now:

1. Fetch a large pool of filtered candidates (up to 10 × 100 = 1,000 items).  
2. Scrape them one by one.  
3. Keep only those that pass the body-text relevance test.  
4. Stop when I reach **up to 30 valid articles** per firm, and then save results to CSV.  
Finally, I combine everything into a single `news_df` dataset.


In [5]:
TARGET_PER_TICKER = 30   # desired number of valid articles per company

all_results = []

for tkr in TICKERS:
    keywords = TICKER_KEYWORDS[tkr]
    time.sleep(0.8)

    # Step A: fetch candidate metadata (already filtered & deduplicated)
    df_candidates = fetch_candidates_for_ticker(
        tkr,
        keywords,
        max_pages=10,
        page_size=100
    )
    if df_candidates.empty:
        continue

    print(f"📰 Scraping articles for {tkr} until we reach {TARGET_PER_TICKER} valid pieces ...")

    valid_rows = []
    for i, row in df_candidates.reset_index(drop=True).iterrows():
        if len(valid_rows) >= TARGET_PER_TICKER:
            break

        url = row["url"]
        print(f"  ({i+1}/{len(df_candidates)}) {url}")
        text_i = scrape_article_text(url)

        if not body_relevant(text_i, tkr):
            print("    ⏭ Skipped: not relevant or too short")
            continue

        row_dict = row.to_dict()
        row_dict["scraped_text"] = text_i
        row_dict["scraped_len"]  = len(text_i)
        valid_rows.append(row_dict)

        time.sleep(1.0)  # be polite to host servers

    df_valid = pd.DataFrame(valid_rows)
    print(f"✅ {tkr}: collected {len(df_valid)} valid scraped articles.")

    df_valid.to_csv(f"results_{tkr}.csv", index=False)
    all_results.append(df_valid)

# Combine across tickers and defensively deduplicate
if all_results:
    news_df = pd.concat(all_results, ignore_index=True)
    news_df["title_norm"] = news_df["title"].str.lower().str.strip()
    news_df = news_df.drop_duplicates(subset=["ticker", "title_norm"], keep="first")
    print("\n✅ Final news_df after global dedup and filters:", news_df.shape)
else:
    news_df = pd.DataFrame()
    print("⚠️ No valid data collected for any ticker.")

from IPython.display import display
display(news_df[["ticker","source_name","title","scraped_len"]].head(10))


🔎 Fetching metadata for AAPL with query: Apple OR AAPL
  Page 1 status: 200
  Raw API articles: 97
  After dropping missing URLs: 97
  After removing bad domains: 89
  After removing non-news/ad-like pages: 79
  After title deduplication: 79
📰 Scraping articles for AAPL until we reach 30 valid pieces ...
  (1/79) https://9to5mac.com/2025/12/05/car-keys-are-coming-to-the-wallet-app-for-13-new-vehicle-brands-soon/
  (2/79) https://nypost.com/2025/12/05/lifestyle/top-2026-budget-travel-hotspots-revealed/
    ⏭ Skipped: not relevant or too short
  (3/79) https://sny.tv/articles/aaron-glenn-confident-jets-brady-cook-nfl-quarterback
    ⏭ Skipped: not relevant or too short
  (4/79) https://exclaim.ca/film/article/hollywood-defends-paul-dano-against-quentin-tarantino-s-insults
    ⏭ Skipped: not relevant or too short
  (5/79) https://nypost.com/2025/12/05/real-estate/architect-frank-gehry-dead-at-age-96/
    ⏭ Skipped: not relevant or too short
  (6/79) https://gizmodo.com/wikipedia-has-its-

,ticker,source_name,title,scraped_len
0,AAPL,9to5Mac,Car Keys are coming to the Wallet app for 13 new vehicle brands soon,1964
1,AAPL,9to5Mac,"Apple TV’s biggest premiere of the year is next week, with key follow-up after",2905
2,AAPL,Giveawayoftheday.com,More Fun Directions HD Lite,3436
3,AAPL,Nextgov,"House Homeland leaders seek briefings from Apple, Google on ICE-tracking apps",2897
4,AAPL,Mjtsai.com,Newstead Replaces Adams and Jackson,5197
5,AAPL,Yahoo Entertainment,"US lawmakers press Google, Apple to remove apps tracking immigration agents",1734
6,AAPL,9to5Mac,"Future iPhone chips might be produced by Intel, per report",2028
7,AAPL,The Verge,The tech world is sleeping on the most exciting Bluetooth feature in years,10765
8,AAPL,Wccftech,Intel Chips Might Power The Non-Pro iPhone 21 In 2028,2235
9,AAPL,9to5Mac,These four Apple products could launch as soon as next month,3123


### 2.4 Data limitation: why some firms still have fewer than 30 articles

Even after expanding the search to up to 10 pages of NewsAPI results per ticker (so up to 1,000 raw items) and using a relatively lenient length threshold (250 characters), some firms still end up with fewer than 30 valid scraped articles.

This is not a coding error but a property of the underlying news supply:

- A non-trivial share of URLs come from paywalled or low-value aggregators such as The Fly, Biztoc, and Slashdot, which I intentionally exclude.  
- Many remaining links are software pages, app listings, or shopping “deals”, which are removed by my non-news and ad-like keyword filters.  
- Some URLs return very little readable `<p>` content (e.g., login walls or short blurbs), and these are dropped by the body-text quality test.

In these cases, I keep all valid articles that pass my filters and treat the smaller sample size as a **data limitation**. I deliberately prioritize text quality and topical relevance over mechanically forcing every firm to have exactly 30 articles, which would dilute the downstream sentiment and topic analysis.


In [6]:
from google.colab import drive
drive.mount('/content/drive')
news_df.to_csv('/content/drive/MyDrive/news_df.csv', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Sentiment Analysis with VADER

After building the cleaned news corpus, I use VADER to measure the tone of each article.
VADER is a rule-based sentiment model that produces a compound score in [-1, 1], which
is particularly suitable for short, informal, and finance-related text.

I apply VADER to the scraped article bodies, then aggregate sentiment to the ticker level,
computing the average sentiment and the share of positive/negative/neutral articles for each firm.
This provides a simple proxy for overall “market tone” toward each company.


In [7]:
# VADER Sentiment for Scraped Articles → Aggregate → Plot

# Install packages (only needed the first time)
!pip -q install --upgrade nltk plotly kaleido

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import plotly.express as px
import plotly.io as pio

# --- IMPORTANT: reset Plotly template/renderer to avoid Template error ---
# Some environments set pio.templates.default incorrectly (to a Template object).
# Here we force it back to a simple named template.
pio.templates.default = "plotly"
px.defaults.template = "plotly"
# In Colab the default renderer is usually fine; no need to override.
# If you really want, you can uncomment the next line:
# pio.renderers.default = "colab"

# Ensure VADER lexicon
nltk.download('vader_lexicon')

# Initialize analyzer
sia = SentimentIntensityAnalyzer()

# Helper to compute VADER compound score in [-1, 1]
def compound_score(text):
    return sia.polarity_scores(text or "")["compound"]

# Ensure scraped_text exists
if "scraped_text" not in news_df.columns:
    raise ValueError("news_df missing 'scraped_text'. Run the scraping step first.")

# 1. Compute article-level VADER sentiment
print("Computing VADER compound sentiment for scraped_text ...")
news_df["vader_compound"] = news_df["scraped_text"].fillna("").apply(compound_score)

# 2. Optional sentiment labels
POS, NEG = 0.05, -0.05
news_df["sentiment_label"] = news_df["vader_compound"].apply(
    lambda c: "pos" if c > POS else ("neg" if c < NEG else "neu")
)

print("\nExample rows:")
display(news_df[["ticker", "title", "vader_compound", "sentiment_label"]].head())

# 3. Aggregate sentiment at the ticker level
ticker_sent = (
    news_df.groupby("ticker")
    .agg(
        mean_vader=("vader_compound", "mean"),
        n_articles=("scraped_text", "count"),
        pct_pos=("sentiment_label", lambda s: (s == "pos").mean()),
        pct_neg=("sentiment_label", lambda s: (s == "neg").mean()),
        pct_neu=("sentiment_label", lambda s: (s == "neu").mean()),
    )
    .reset_index()
)

# Round for readability
ticker_sent["mean_vader"] = ticker_sent["mean_vader"].round(3)
ticker_sent["pct_pos"] = (ticker_sent["pct_pos"] * 100).round(1)
ticker_sent["pct_neg"] = (ticker_sent["pct_neg"] * 100).round(1)
ticker_sent["pct_neu"] = (ticker_sent["pct_neu"] * 100).round(1)

print("\nTicker-level sentiment summary:")
display(ticker_sent)

# 4. Visualization: bar chart of average sentiment by ticker
fig = px.bar(
    ticker_sent,
    x="ticker",
    y="mean_vader",
    color="mean_vader",
    color_continuous_scale="RdYlGn",
    hover_data=["n_articles", "pct_pos", "pct_neg", "pct_neu"],
    title="Average VADER Sentiment by Ticker (Scraped News Articles)",
    labels={
        "mean_vader": "Average Sentiment (Compound)",
        "ticker": "Stock Ticker",
    },
)

fig.update_layout(
    xaxis_title="Ticker",
    yaxis_title="Mean Compound Sentiment",
    title_font_size=18,
    template="plotly",  # explicitly set a safe template here as well
)

fig.show()

# 5. Save HTML and PNG using Kaleido
fig.write_html("ticker_sentiment.html", include_plotlyjs="cdn", full_html=True)
fig.write_image("ticker_sentiment.png")

print("Saved: ticker_sentiment.html and ticker_sentiment.png")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Computing VADER compound sentiment for scraped_text ...

Example rows:


,ticker,title,vader_compound,sentiment_label
0,AAPL,Car Keys are coming to the Wallet app for 13 new vehicle brands soon,0.9913,pos
1,AAPL,"Apple TV’s biggest premiere of the year is next week, with key follow-up after",0.9965,pos
2,AAPL,More Fun Directions HD Lite,0.9957,pos
3,AAPL,"House Homeland leaders seek briefings from Apple, Google on ICE-tracking apps",0.9917,pos
4,AAPL,Newstead Replaces Adams and Jackson,0.9975,pos



Ticker-level sentiment summary:


,ticker,mean_vader,n_articles,pct_pos,pct_neg,pct_neu
0,AAPL,0.965,15,100.0,0.0,0.0
1,GS,0.930,30,96.7,3.3,0.0
2,JPM,0.808,23,91.3,8.7,0.0
3,MSFT,0.534,20,75.0,25.0,0.0
4,NVDA,0.701,30,86.7,13.3,0.0
5,TSLA,0.506,30,76.7,20.0,3.3
6,XOM,0.621,30,80.0,20.0,0.0


BrowserDepsError: It seems like you are running a slim version of your operating system and are missing some common dependencies. The following command should install the required dependencies on most systems:

$ sudo apt update && sudo apt-get install libnss3 libatk-bridge2.0-0 libcups2 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libxkbcommon0 libpango-1.0-0 libcairo2 libasound2

If you have already run the above command and are still seeing this error, or the above command fails, consult the Kaleido documentation for operating system to install chromium dependencies.

For support, run the command `choreo_diagnose` and create an issue with its output.

### **Figure: Average VADER Sentiment by Ticker — Interpretation**

The figure shows the average sentiment score for news articles associated with each company. Apple, Goldman Sachs, and JPMorgan receive the most positive tone, indicating strong market confidence. Tech firms such as Microsoft, Tesla, and NVIDIA show more mixed sentiment, reflecting uncertainty around AI growth and regulatory pressure. ExxonMobil sits in the middle, consistent with the typically balanced tone of energy-sector news. These cross-industry differences set up the clustering patterns seen later in the two-factor “Sentiment × Topic Exposure” analysis.

## 4. Text Cleaning & Preprocessing

Before topic modeling, all scraped article text is normalized so that TF-IDF reflects meaningful economic language rather than noise. The pipeline includes lowercasing, URL removal, tokenization, stopword filtering, and lemmatization. I also removed company names, advertisement boilerplate, and repetitive footer text to avoid contaminating topic exposure and prevent artificial clustering driven by firm-specific jargon.


In [9]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

english_stopwords = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# Remove company names to avoid contaminating thematic signals
ticker_words = [
    "aapl","apple","msft","microsoft","nvda","nvidia",
    "tsla","tesla","xom","exxon","mobil","gs","goldman",
    "sachs","jpm","jpmorgan","morgan"
]

# text cleaning
def clean_text(text):
    if text is None:
        return ""

    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)

    tokens = nltk.word_tokenize(text)

    tokens = [tok for tok in tokens if tok not in ticker_words]
    tokens = [tok for tok in tokens if tok not in english_stopwords and len(tok) > 2]
    tokens = [lemmatizer.lemmatize(tok) for tok in tokens]

    return " ".join(tokens)

news_df["scraped_text_clean"] = news_df["scraped_text"].astype(str).apply(clean_text)
news_df.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


ticker           source_name  \
0   AAPL               9to5Mac   
1   AAPL               9to5Mac   
2   AAPL  Giveawayoftheday.com   
3   AAPL               Nextgov   
4   AAPL            Mjtsai.com   

                                                                            title  \
0            Car Keys are coming to the Wallet app for 13 new vehicle brands soon   
1  Apple TV’s biggest premiere of the year is next week, with key follow-up after   
2                                                     More Fun Directions HD Lite   
3   House Homeland leaders seek briefings from Apple, Google on ICE-tracking apps   
4                                             Newstead Replaces Adams and Jackson   

                                                                                                                                                                                                                                                            description  \
0                                  Apple debuted some great new CarPlay features in iOS 26, but the company has more coming for your vehicle soon. Here are the 13 new vehicle brands that Apple says will soon support car keys in the Apple Wallet app.\n\n\n\n more…   
1  Apple TV has had a big year already, but next week brings its most high-profile premiere yet: F1 The Movie joins the streaming service. There’s also a key follow-up coming to Apple TV soon with the Formula 1 sport’s arrival. Here are the details.\n\n\n\n more…   
2                                                                                                If you liked our “Fun with Directions” app, you’ll also love More Fun with Directions Lite too! Developed by a speech pathologist, this app continues with the same...   
3                                                                                                                         Republican lawmakers say crowdsourced tools that flag immigration enforcement activity may endanger federal personnel and disrupt operations.   
4  Apple (MacRumors): Apple today announced that Jennifer Newstead will become Apple’s general counsel on March 1, 2026, following a transition of duties from Kate Adams, who has served as Apple’s general counsel since 2017. She was previously chief legal office…   

                                                                                                                   url  \
0                 https://9to5mac.com/2025/12/05/car-keys-are-coming-to-the-wallet-app-for-13-new-vehicle-brands-soon/   
1         https://9to5mac.com/2025/12/05/apple-tvs-biggest-premiere-of-the-year-is-next-week-with-key-follow-up-after/   
2                                                     https://iphone.giveawayoftheday.com/more-fun-directions-hd-lite/   
3  https://www.nextgov.com/policy/2025/12/house-homeland-leaders-seek-briefings-apple-google-ice-tracking-apps/409981/   
4                                              https://mjtsai.com/blog/2025/12/05/newstead-replaces-adams-and-jackson/   

            publishedAt  \
0  2025-12-05T21:57:11Z   
1  2025-12-05T21:38:58Z   
2  2025-12-05T21:22:33Z   
3  2025-12-05T21:15:00Z   
4  2025-12-05T21:07:27Z   

                                                                       title_norm  \
0            car keys are coming to the wallet app for 13 new vehicle brands soon   
1  apple tv’s biggest premiere of the year is next week, with key follow-up after   
2                                                     more fun directions hd lite   
3   house homeland leaders seek briefings from apple, google on ice-tracking apps   
4                                             newstead replaces adams and jackson   

                                                                      title_lower  \
0            car keys are coming to the wallet app for 13 new vehicle brands soon   
1  apple tv’s biggest premiere of the year is next week, with key follow-up after   
2                    

## 5. Dictionary-Based Topic Modeling (Macro, AI, Energy)

Rather than using LDA—which is noisy, random, and hard to interpret—I apply a dictionary-based topic model. This approach is deterministic, transparent, and fully customizable: each topic is defined by curated keyword lists representing macro, AI, and energy themes. The method aligns with industry practice (e.g., Bloomberg and FactSet keyword tagging) and produces stable, interpretable exposure scores for each company. Below are the lists of words I prepared for better topic modeling.


In [10]:
topic_macro = [
    "inflation","fed","interest","rate","rates","growth","economy","economic",
    "recession","employment","jobs","gdp","policy","central","market","markets",
    "bond","bonds","yield","credit","debt","tightening","easing"
]

topic_ai = [
    "ai","artificial","intelligence","model","models","algorithm","compute",
    "computing","semiconductor","semiconductors","chip","chips","gpu","gpus",
    "cloud","neural","deep","training","automation","data","machine","learning"
]

topic_energy = [
    "oil","gas","energy","barrel","production","opec","supply","demand",
    "commodity","refinery","natural","pipeline","petroleum","crude","brent","wti"
]

topic_dicts = {
    "macro": topic_macro,
    "ai": topic_ai,
    "energy": topic_energy
}


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Build TF-IDF matrix on cleaned text
tfidf_vec = TfidfVectorizer(stop_words="english", max_df=0.85, min_df=2)
X = tfidf_vec.fit_transform(news_df["scraped_text_clean"].fillna(""))

# Create vocabulary index for fast lookup
vocab = tfidf_vec.get_feature_names_out()
vocab_index = {w: i for i, w in enumerate(vocab)}

# Initialize topic exposure container
topic_scores = {"ticker": news_df["ticker"]}

# Compute topic exposure by summing TF-IDF values of dictionary words
for name, words in topic_dicts.items():
    idx = [vocab_index[w] for w in words if w in vocab_index]
    exposure = X[:, idx].sum(axis=1).A.flatten() if idx else np.zeros(X.shape[0])
    topic_scores[name] = exposure

# Convert to DataFrame
topic_exposure_df = pd.DataFrame(topic_scores)

# Aggregate exposures to ticker level
ticker_topic_exposure = (
    topic_exposure_df.groupby("ticker")[["macro","ai","energy"]].mean().reset_index()
)

ticker_topic_exposure

,ticker,macro,ai,energy
0,AAPL,0.020907,0.101012,0.004990
1,GS,0.262090,0.116873,0.095621
2,JPM,0.182558,0.057854,0.024626
3,MSFT,0.065900,0.198542,0.020777
4,NVDA,0.072572,0.200131,0.029121
5,TSLA,0.050564,0.058546,0.048937
6,XOM,0.164322,0.043490,0.449258


## 6. Combining Sentiment + Topic Exposure

I merge firm-level VADER sentiment with dictionary-based topic exposure to create a two-factor view of each company. Topic exposure is computed by summing TF-IDF weights for macro, AI, and energy keywords, then aggregating to the ticker level.

Combining these scores with average sentiment produces a matrix linking **market tone** and **policy relevance**. Plotting the two dimensions reveals how firms cluster across themes, showing which sectors attract positive narratives and which are tied to policy-sensitive topics.


In [12]:
# Merge sentiment and topic exposure into a single DataFrame
twofactor_df = ticker_sent.merge(ticker_topic_exposure, on="ticker", how="inner")
display(twofactor_df)

import plotly.express as px

# Scatter: sentiment vs. macro exposure
px.scatter(
    twofactor_df, x="macro", y="mean_vader", text="ticker",
    size="n_articles", color="mean_vader",
    color_continuous_scale="RdYlGn",
    title="Sentiment × Macro Topic Exposure"
).show()

# Scatter: sentiment vs. AI exposure
px.scatter(
    twofactor_df, x="ai", y="mean_vader", text="ticker",
    size="n_articles", color="mean_vader",
    color_continuous_scale="RdYlGn",
    title="Sentiment × AI Topic Exposure"
).show()

# Scatter: sentiment vs. energy exposure
px.scatter(
    twofactor_df, x="energy", y="mean_vader", text="ticker",
    size="n_articles", color="mean_vader",
    color_continuous_scale="RdYlGn",
    title="Sentiment × Energy Topic Exposure"
).show()

,ticker,mean_vader,n_articles,pct_pos,pct_neg,pct_neu,macro,ai,energy
0,AAPL,0.965,15,100.0,0.0,0.0,0.020907,0.101012,0.004990
1,GS,0.930,30,96.7,3.3,0.0,0.262090,0.116873,0.095621
2,JPM,0.808,23,91.3,8.7,0.0,0.182558,0.057854,0.024626
3,MSFT,0.534,20,75.0,25.0,0.0,0.065900,0.198542,0.020777
4,NVDA,0.701,30,86.7,13.3,0.0,0.072572,0.200131,0.029121
5,TSLA,0.506,30,76.7,20.0,3.3,0.050564,0.058546,0.048937
6,XOM,0.621,30,80.0,20.0,0.0,0.164322,0.043490,0.449258


## 7. Interpretation of Topic Exposure × Sentiment Results

### 7.1 Macro Topic Exposure

Across the sample, macro exposure remains relatively low for all companies, but the variations help explain differences in how firms are discussed in news coverage.

* **Goldman Sachs (GS)** shows **high sentiment and slightly elevated macro exposure**, reflecting frequent mentions of markets, interest rates, and monetary policy — consistent with its role in global finance.
* **JPMorgan (JPM)** also exhibits **positive sentiment with moderate macro references**, likely driven by commentary on interest rates, deposits, and consumer credit.
* **Tech firms (AAPL, MSFT, NVDA, TSLA)** cluster toward **lower macro exposure**, meaning their coverage is dominated more by product cycles and innovation rather than macroeconomics. Among them, **Tesla (TSLA)** shows the **lowest sentiment**, reflecting more negative news tone despite low macro discussion.

**Interpretation:**
Financial firms naturally appear in macroeconomic narratives and receive more positive tone, while tech firms’ sentiment seems unrelated to macro exposure—suggesting their sentiment variation is driven by sector-specific factors, not by the macro cycle.


### 7.2 AI Topic Exposure

AI exposure varies widely across firms and produces a clear pattern:

* **Nvidia (NVDA)** and **Microsoft (MSFT)** show **high AI exposure**, aligning with their roles in GPUs and cloud AI ecosystems.

  * NVDA has moderately positive sentiment; MSFT’s sentiment is more neutral–negative, reflecting issues like AI safety, regulation, or competitive pressures.
* **Apple (AAPL)** and **Goldman Sachs (GS)** have **moderate AI exposure but very high sentiment**, meaning AI-related discussions for these firms tend to be positive or peripheral.
* **Tesla (TSLA)** has **low AI exposure and the lowest sentiment**, suggesting that recent Tesla coverage is driven more by operational and management-related narratives than AI.

**Interpretation:**
AI exposure does **not** guarantee positive sentiment. Instead, it appears that firms at the center of the AI boom experience both optimism and skepticism, while firms with only peripheral AI mentions enjoy more positive tone. This supports concerns that markets may be increasingly sensitive to an **AI “hype cycle”** — a potential policy-relevant signal.


### 7.3 Energy Topic Exposure

Energy exposure sharply distinguishes ExxonMobil (XOM) from the rest:

* **XOM** shows **very high energy exposure**, as expected, with mixed sentiment reflecting volatile oil markets, production constraints, and policy debates about climate transitions.
* All other firms cluster near zero, indicating that energy-related language does not appear in their coverage in a meaningful way.

Sentiment among non-energy firms thus reflects factors unrelated to the energy sector.

**Interpretation:**
The energy topic is highly concentrated in XOM, confirming the validity of dictionary-based exposure. Its moderate sentiment reinforces that commodity-driven industries receive more policy-charged, less uniformly positive coverage.


### 7.4 Cross-Topic Insight

Taken together, the topic–sentiment scatterplots reveal clear patterns across sectors.

Financial firms exhibit both high sentiment and moderate macro exposure, suggesting confidence in financial stability narratives and generally positive news tone around banks.

In contrast, tech firms show lower sentiment combined with widely varying levels of AI exposure, indicating early skepticism about the speed and sustainability of AI-driven growth despite heavy media attention.

Energy-related exposure is concentrated almost entirely in ExxonMobil, whose sentiment reflects the volatility of commodity cycles and ongoing climate policy tensions rather than broader market narratives.


## 8. Policy Implication

The patterns from section 7 also carry important policy implications.

The relatively **high sentiment toward financial institutions** suggests that markets currently view the macroeconomic environment—interest rates, liquidity conditions, and regulatory stability—as predictable and well-managed. **Policymakers should continue monitoring systemic risk but may not need immediate intervention.**

In contrast, the **lower sentiment attached to major tech firms**, even as their exposure to AI-related language rises, signals growing public uncertainty about the pace of AI deployment, data governance, and the potential for over-investment in emerging technologies. **This highlights a policy need for clearer AI regulation, transparency standards, and competition oversight to prevent bubbles and protect consumers.**

Meanwhile, the **energy sector’s concentrated topic exposure**—centered almost entirely on ExxonMobil—shows that sentiment is closely tied to commodity cycles and climate-related debates. **Policymakers should anticipate that shifts in environmental regulation, carbon pricing, or geopolitical energy events will disproportionately influence market expectations for this sector.**

Overall, the joint topic–sentiment results point toward a regulatory environment in which **AI governance**, **financial stability**, and **energy transition policy** are increasingly interconnected and influential for market confidence.

## 9. Summary of Project and Key Innovations

This project examines how news sentiment differs across seven major U.S. companies and how this sentiment interacts with policy-relevant themes such as macroeconomics, artificial intelligence, and energy. I built a full pipeline that collects, cleans, analyzes, and visualizes real financial news using web scraping, VADER sentiment analysis, TF-IDF weighting, and custom topic modeling.

A key innovation is the **enhanced data-collection system**: expanding search keywords beyond tickers, multi-pass scraping to guarantee 30 usable articles per firm, filtering out irrelevant pages and low-quality domains (e.g., paywalls, ads), and removing duplicated headlines. This produces a much higher-quality dataset than standard API queries.

In preprocessing, I added **domain-specific cleaning**—removing company names to avoid topic leakage, applying custom stopwords, and filtering articles heuristically. This ensures cleaner linguistic signals and avoids artificially inflated topic scores.

Methodologically, I moved beyond LDA and implemented a **dictionary-based topic model** with three policy-aligned themes (macro, AI, energy). This approach is stable, interpretable, and consistent with how professional financial data vendors classify news.

Finally, I combined sentiment and topic exposure into **two-factor scatterplots**, revealing clear sector patterns: financial firms cluster with strong sentiment and higher macro relevance, while tech firms show weaker sentiment despite high AI exposure.

Together, these innovations—**improved scraping, targeted filtering, customized cleaning, policy-aligned topic modeling, and two-factor visualization**—form a robust, transparent, and policy-relevant text-analysis pipeline.